In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from astropy.table import Table

In [2]:
apt_pid = 998
PID_MAP = {981: "GPS", 994: "HLWAS", 998: "GBTDS", 999: "HLTDS"}

In [3]:
joint_df = Table.read(f"roman_opsim/joint_schedule_apt_pid{apt_pid}.ecsv")
joint_df = joint_df.to_pandas()

In [4]:
# Group by PASS, sorted chronologically
pass_groups = [
    grp for _, grp in
    joint_df.sort_values(['mjd_start', 'OBSERVATION', 'VISIT']).groupby('PASS', sort=False)
]

# One frame per pass — no per-exposure expansion
frame_list = pass_groups

n_frames = len(frame_list)
print(f"Total frames: {n_frames}  (passes: {len(pass_groups)})")

# --- Filter color palette (wavelength-ordered, colorblind-friendly Paul Tol colors) ---
BP_COLORS = {
    'F062': '#4477AA',
    'F087': '#66CCEE',
    'F106': '#228833',
    'F129': '#CCBB44',
    'F158': '#EE6677',
    'F184': '#AA3377',
    'F213': '#BBBBBB',
    'W146': '#EE8866',
    'PRISM': "#755903",
    'GRISM': "#75071F",
}

fig, ax = plt.subplots(1, 1, figsize=(8, 6))

ax.set_xlabel("RA (deg)")
ax.set_ylabel("DEC (deg)")
ax.set_xlim(joint_df['RA'].max() + 0.5, joint_df['RA'].min() - 0.5)  # RA decreases left→right
ax.set_ylim(joint_df['DEC'].min() - 0.5, joint_df['DEC'].max() + 0.5)

_box = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='0.7')
ax.text(0.5, 0.97, f"PID {apt_pid} ({PID_MAP.get(apt_pid)})",
        transform=ax.transAxes, ha='center', va='top', fontsize=10, bbox=_box)

# Filter legend
bp_in_data = sorted(joint_df['BANDPASS'].unique())
patches = [mpatches.Patch(color=BP_COLORS.get(bp, 'steelblue'), label=bp) for bp in bp_in_data]
ax.legend(handles=patches, loc='lower left', title='Filter', fontsize=8, title_fontsize=8, framealpha=0.8)

scat = ax.scatter([], [], s=20, alpha=0.5, edgecolors='none')
cur  = ax.scatter([], [], s=40, alpha=0.9, edgecolors='white', linewidths=0.5)

time_label = ax.text(0.5, 0.03, '', transform=ax.transAxes,
                     fontsize=9, ha='center', va='bottom')

shown_xy, shown_cols = [], []
EMPTY = np.empty((0, 2))

def update(i):
    if i == 0:
        return scat, cur, time_label

    g = frame_list[i - 1]
    pts = list(zip(g['RA'], g['DEC']))
    pass_id = g['PASS'].iloc[0]
    colors = [BP_COLORS.get(bp, 'steelblue') for bp in g['BANDPASS']]

    shown_xy.extend(pts)
    shown_cols.extend(colors)
    scat.set_offsets(shown_xy)
    scat.set_facecolor(shown_cols)
    cur.set_offsets(pts)
    cur.set_facecolor(colors)

    time_label.set_text(f"Pass {pass_id}   MJD {g['mjd_start'].iloc[0]:.1f}")

    return scat, cur, time_label

plt.rcParams['animation.embed_limit'] = 100  # MB
ani = FuncAnimation(fig, update, frames=n_frames + 1, interval=100, blit=True)
plt.close(fig)

# save animation as mp4 (requires ffmpeg)
ani.save(f'animations/pid_{apt_pid}_coverage.mp4', writer='ffmpeg', dpi=150)

Total frames: 100  (passes: 100)
